In [1]:
pip install torch monai nibabel numpy matplotlib


Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
import monai
print ("Torch version:", torch.__version__)
print ("MONAI version:", monai.__version__)
print ("CUDA available:", torch.cuda.is_available())

Torch version: 2.5.1
MONAI version: 1.5.0
CUDA available: False


In [4]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from monai.networks.nets import UNet

# ---------- helpers ----------
def ensure_dir(path="outputs"):
    os.makedirs(path, exist_ok=True)
    return path

def get_device(prefer_gpu=True):
    if prefer_gpu and torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

def load_synthetic_volume(shape=(1, 1, 128, 128, 128), seed=7):
    g = torch.Generator().manual_seed(seed)
    return torch.randn(shape, generator=g)

def preprocess(volume: torch.Tensor):
    if volume is None:
        raise ValueError("Input volume is None.")
        if not isinstance(volume, torch.Tensor):
            raise TypeError("Input volume must be a torch.Tensor.")
            if volume.ndim != 5:
                raise ValueError(f"Invalid volume shape {tuple(volume.shape)}. Expected 5D (B,C,H,W,D).")
    volume = volume.float()
    mean = volume.mean()
    std = volume.std()
    if std == 0:
        raise ValueError("Volume std is 0; cannot normalize.")
    return (volume - mean) / (std + 1e-8)

def build_model(device):
    model = UNet(
        spatial_dims=3,
        in_channels=1,
        out_channels=2,
        channels=(16, 32, 64),
        strides=(2, 2),
        num_res_units=1,
    ).to(device)
    model.eval()
    return model

@torch.no_grad()
def infer(model, volume, device):
    volume = volume.to(device)
    return model(volume)

def logits_to_mask(logits):
    if logits.ndim != 5 or logits.shape[1] < 2:
        raise ValueError(f"Invalid logits shape {tuple(logits.shape)}. Expected (B,2,H,W,D).")
    return torch.argmax(logits, dim=1) # (B,H,W,D)

def save_overlay_png(volume, mask, out_path):
    vol = volume[0, 0].detach().cpu().numpy()
    m = mask[0].detach().cpu().numpy().astype(np.int32)
    mid = vol.shape[-1] // 2
    
    plt.figure()
    plt.imshow(vol[:, :, mid], cmap="gray")
    plt.imshow(m[:, :, mid], alpha=0.35)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight", pad_inches=0)
    plt.close()

def run_pipeline(volume=None, out_dir="outputs"):
    ensure_dir(out_dir)
    try:
        # TC-01 Load
        if volume is None:
            volume = load_synthetic_volume()
            source = "synthetic"
        else:
            source = "provided"
            print(f"[TC-01] Loaded input ({source}) shape={tuple(volume.shape)} dtype={volume.dtype}")
            
        # TC-02 Preprocess
        volume_p = preprocess(volume)
        print(f"[TC-02] Preprocessed shape={tuple(volume_p.shape)} mean={volume_p.mean().item():.4f} std={volume_p.std().item():.4f}")
        
        # TC-03 Device selection
        device = get_device()
        print(f"[TC-03] Device selected: {device} (cuda_available={torch.cuda.is_available()})")
        
        # TC-04 Inference
        model = build_model(device)
        logits = infer(model, volume_p, device)
        print(f"[TC-04] Inference logits shape={tuple(logits.shape)}")
        
        # TC-05 Mask
        mask = logits_to_mask(logits)
        uniq = torch.unique(mask.cpu()).tolist()
        print(f"[TC-05] Mask shape={tuple(mask.shape)} unique_values={uniq}")
        
        # TC-07 Save mask
        mask_path = os.path.join(out_dir, "mask.pt")
        torch.save(mask.cpu(), mask_path)
        print(f"[TC-07] Saved mask: {mask_path}")
        
        # TC-06 Save overlay image
        overlay_path = os.path.join(out_dir, "overlay.png")
        save_overlay_png(volume_p.cpu(), mask.cpu(), overlay_path)
        print(f"[TC-06] Saved overlay: {overlay_path}")
        
        # TC-07 Confirm persistence
        exists = os.path.exists(mask_path) and os.path.exists(overlay_path)
        print(f"[TC-07] Output files exist after execution: {exists}")
        return {"mask_path": mask_path, "overlay_path": overlay_path, "device": str(device)}
    except Exception as e:
        # TC-08 Invalid input handling
        print(f"[TC-08] ERROR (handled gracefully): {type(e).__name__}: {e}")
        return {"error": f"{type(e).__name__}: {e}"}

# ---- Run happy path (TC-01 to TC-07) ----
result = run_pipeline()
result

[TC-02] Preprocessed shape=(1, 1, 128, 128, 128) mean=0.0000 std=1.0000
[TC-03] Device selected: cpu (cuda_available=False)
[TC-04] Inference logits shape=(1, 2, 128, 128, 128)
[TC-05] Mask shape=(1, 128, 128, 128) unique_values=[0, 1]
[TC-07] Saved mask: outputs\mask.pt
[TC-06] Saved overlay: outputs\overlay.png
[TC-07] Output files exist after execution: True


{'mask_path': 'outputs\\mask.pt',
 'overlay_path': 'outputs\\overlay.png',
 'device': 'cpu'}

In [6]:
bad_volume = torch.rand(128, 128)
run_pipeline(volume=bad_volume)

[TC-01] Loaded input (provided) shape=(128, 128) dtype=torch.float32
[TC-02] Preprocessed shape=(128, 128) mean=-0.0000 std=1.0000
[TC-03] Device selected: cpu (cuda_available=False)
[TC-08] ERROR (handled gracefully): RuntimeError: Expected 4D (unbatched) or 5D (batched) input to conv3d, but got input of size: [128, 128]


{'error': 'RuntimeError: Expected 4D (unbatched) or 5D (batched) input to conv3d, but got input of size: [128, 128]'}

In [1]:
import os
os.getcwd()

'C:\\Users\\jkima'